In [ ]:
import os
os.chdir(Insert Project Path)

## Notebook 2: Database
This notebook focused on using functions from *database.py* to finish the API-to-database pipeline. In *schema.sql* I laid the schema out for the close approach table, and wanted to begin inputting the data extracted from the NASA NeoWs API.

In [ ]:
from src.api import asteroid_data_extract, rec_to_df, fetch_dates_asteroids, fetch_daterange_asteroids
from src.processing import standardise_asteroid_data, asteroid_data_report, validate_report
from src.database import insert_events, get_dates, get_period_dates

In [ ]:
#checking table 'close_approaches' exists
import sqlite3
c = sqlite3.connect("data/nasa_asteroids.db")
cursor = c.cursor()
cursor.execute("""
    SELECT name
    FROM sqlite_master
    WHERE type = 'table';
""")

print(cursor.fetchall())
c.close()

At first, I was having to manually check the report before finally inserting the data into the database. I wanted this to happen automatically, and flag an error if one were to occur. This was especially important as the database would be taking in 14000+ rows of data, and API rate limits meant that errors could occur without data being invalid: I wanted to know one from the other. \
I began by manually inserting one day, and then one month into the database, so I could check if anything went wrong. After, I automated the process so I could get thousands of rows of data cleaned, standardised, and validated before being placed in the database.

In [ ]:
#testing insertion for one date 2020-01-01, if report is okay insert
asteroids = fetch_dates_asteroids("2020-01-01")
rec = asteroid_data_extract(asteroids)
df = rec_to_df(rec)
sdf = standardise_asteroid_data(df)
report = asteroid_data_report(sdf)
print(df)

In [ ]:
insert_events(sdf)

In [ ]:
import sqlite3
#this checks the number of rows - useful for ensuring correct methods
with sqlite3.connect("data/nasa_asteroids.db") as c:

    cursor = c.cursor()

    cursor.execute("""
        SELECT COUNT(*)
        FROM close_approaches;
    """)

    print(cursor.fetchone())

In [ ]:
#testing api extraction and database insertion on month of january 2020
start_date, end_date = "2020-01-01", "2020-01-31"
jan2020df = fetch_daterange_asteroids(start_date, end_date)
jan2020sdf = standardise_asteroid_data(jan2020df)
j2020report = asteroid_data_report(jan2020sdf)
print(report)

In [ ]:
#passed report so insert
insert_events(jan2020sdf)

### API to SQL
Below is the code used to extract all asteroid data from NASA NeoWs API, and insert into the SQLite database. I did this in 6 month batches to avoid hitting the API request limits, utilising pythons *calendar* library to automate the process, as before my function could only take in specific dates. The historical database runs from 01/01/2020 to 31/07/2026, with plans to automate the process going forward. 

In [ ]:
#pipeline, edit dates for entry
start_year = 2026
start_month = 7
end_year = 2026
end_month = 8

batch_dates = get_period_dates(start_month, start_year, end_month, end_year)
print(batch_dates)

In [ ]:
#inserts events from the above period into SQLite database
for start_date, end_date in batch_dates:
    month_df = fetch_daterange_asteroids(start_date, end_date)
    month_sdf = standardise_asteroid_data(month_df)
    month_report = asteroid_data_report(month_sdf)
    validate_report(month_report)
    insert_events(month_sdf)

Finally, I wanted to verify that the rows were all unique, and corresponded to unique close approach events before beginning to analyse the data.

In [ ]:
#verifying dataset before EDA, for dupes and date range
with sqlite3.connect("data/nasa_asteroids.db") as c:

    result = c.execute("""
        SELECT
            COUNT(*) AS total_rows,
            COUNT(DISTINCT event_id) AS unique_event_ids,
            MIN(close_approach_date) AS first_date,
            MAX(close_approach_date) AS last_date
        FROM close_approaches;
    """)

    print(result.fetchone())